In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
β = 2.5
λ = 0.6

ξ_d = 3.3
ν_d = 1.96

Q = Inf
t_max=1000.0

x_0 = .5
p_0 = .5

dt_save = 2pi/ν_d
t_save = 0.0 : dt_save :t_max

system = SystemMichiel(β, λ, x_0, p_0, 1)
environment = MeanField_Michiel(Q)
drive = DriveOneMode(ξ_d, ν_d, -pi/2)
starting_points_list = [(.5, .5), (1,1), (5,1), (1,5), (6,0),(8,0), (9,0), (10,0)]
p0=plot()
for starting_point in starting_points_list
    x_0, p_0 = starting_point
    system = SystemMichiel(β, λ, x_0, p_0, 1)
    x, p, _ = simulator(t_max, system, drive, environment; t_save)
    scatter!(x, p, label="", ms=2)
end
plot!(xlabel = L"x", ylabel= L"p",
title=L"\beta="*"$β")
display(p0)

In [ ]:
β = 0.5
λ = 0.6

ξ_d = 3.3
ν_d = 1.96

Q = Inf
t_max=1000.0

x_0 = .5
p_0 = .5

dt_save = 2pi/ν_d
t_save = 0.0 : dt_save :t_max

system = SystemMichiel(β, λ, x_0, p_0, 1)
environment = MeanField_Michiel(Q)
drive = DriveOneMode(ξ_d, ν_d, -pi/2)
starting_points_list = [(.5, .5), (1,1), (5,1), (1,5), (6,0),(8,0), (9,0), (10,0), (-2,-2), (-1.7,-1.7)]
p0=plot()
for starting_point in starting_points_list
    x_0, p_0 = starting_point
    system = SystemMichiel(β, λ, x_0, p_0, 1)
    x, p, _ = simulator(t_max, system, drive, environment; t_save)
    scatter!(x, p, label="", ms=2)
end
plot!(xlabel = L"x", ylabel= L"p",
title=L"\beta="*"$β")
display(p0)

In [ ]:
β = 0.5
λ = 0.6

ξ_d = 3.3
ν_d = 1.96

Q = Inf
t_max=1000.0

x_0 = .5
p_0 = .5

dt_save = 2pi/ν_d*2
t_save = 0.0 : dt_save :t_max

system = SystemMichiel(β, λ, x_0, p_0, 1)
environment = MeanField_Michiel(Q)
drive = DriveOneMode(ξ_d, ν_d, -pi/2)
starting_points_list = [(.5, .5), (1,1), (5,1), (1,5), (6,0),(8,0), (9,0), (10,0)]
p0=plot()
for starting_point in starting_points_list
    x_0, p_0 = starting_point
    system = SystemMichiel(β, λ, x_0, p_0, 1)
    x, p, _ = simulator(t_max, system, drive, environment; t_save)
    scatter!(x, p, label="", ms=2)
end
plot!(xlabel = L"x", ylabel= L"p",
title=L"\beta="*"$β")
display(p0)

In [ ]:
β = 0.5
λ = 0.6

ξ_d = 3.3
ν_d = 1.96

Q = Inf
t_max=1000.0

x_0 = .5
p_0 = .5

dt_save = 2pi/ν_d
t_save = 0.0 : dt_save :t_max

starting_points_list = [(.5, .5), (1.0,1.0), (5.0,1.0), (1.0,5.0), (6.0,0.0), (8.0,0.0), (9.0,0.0), (10.0,0.0)]
p0=plot()
for starting_point in starting_points_list
    x_0, p_0 = starting_point
    system = SystemMichiel(β, λ, 10.0, 0.0, 1)
    system_prepared = prepare(system)
    drive = DriveOneMode(ξ_d, ν_d, -pi/2)
    drive_prepared = prepare(drive)
    environment = MeanField_Michiel(Q)
    environment_prepared = prepare(system,environment)

    tspan = (0.0, t_max)
    dt = get_dt(system, environment)

    params = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    u_0 = [p_0, x_0]
    prob = ODEProblem(equation_1ODE!, u_0, tspan, params)

    sol = solve(prob, SSPRK22(),dt=dt,
        saveat = t_save
        )

    p = getindex.(sol.u,1) 
    x = getindex.(sol.u,2)


    scatter!(x, p, label="", ms=2)
end
plot!(xlabel = L"x", ylabel= L"p",
title=L"\beta="*"$β")
display(p0)

# Comparison with the change of frame

In [ ]:
β = 0.5
λ = 0.6

ξ_d = 3.3
ν_d = 1.96

Q = Inf
t_max=1000.0

x_0 = .5
p_0 = .5

dt_save = 2pi/ν_d
t_save = 0.0 : dt_save :t_max

system = SystemMichiel(β, λ, x_0, p_0, 1)
environment = MeanField_Michiel(Q)
drive = DriveOneMode(ξ_d, ν_d, 0.0)
starting_points_list = [(.5, .5), (1,1), (5,1), (1,5), (6,0),(8,0), (9,0), (10,0)]
p0=plot()
for starting_point in starting_points_list
    print(starting_point)
    x_0, p_0 = starting_point
    x_0, p_0 = (x_0 + p_0)/sqrt(2), (x_0 - p_0)/sqrt(2)
    system = SystemMichiel(β, λ, x_0, p_0, 1)
    x, p, _ = simulator(t_max, system, drive, environment; t_save)
    scatter!(x, p, label="", ms=2)
end
plot!(xlabel = L"x", ylabel= L"p",
title=L"\beta="*"$β")
display(p0)


In [ ]:
#Michiel's system
function test_equation!(ddu, du, u, p::SimulationParams{SystemMichiel_prepared, D, MeanField_Michiel}, t) where D<:DriveOneMode_prepared
    β, λ = p.system_prepared.system.β, p.system_prepared.system.λ
    E_d, ν_d, phase =  p.drive_prepared.drive.ϵ, p.drive_prepared.drive.ω, p.drive_prepared.drive.phase
    x = u[1]
    ddu[1] = - x + p.drive_prepared.drive_func(E_d*ν_d, ν_d, t, phase) - β/(sqrt(2)*λ)*sin(sqrt(2)*λ*x)
    return(nothing)
end

β = 0.5
λ = 0.6

ξ_d = 3.3
ν_d = 1.96

Q = Inf
t_max=1000.0

x_0 = .5
p_0 = .5

dt_save = 2pi/ν_d
t_save = 0.0 : dt_save :t_max

system = SystemMichiel(β, λ, x_0, p_0, 1)
environment = MeanField_Michiel(Q)
drive = DriveOneMode(E_d, ν_d, 0.0)
starting_points_list = [(.5, .5), (1,1), (5,1), (1,5), (6,0),(8,0), (9,0), (10,0)]
p0=plot()
for starting_point in starting_points_list
    print(starting_point)
    x_0, p_0 = starting_point
    x_0, p_0 = (x_0 + p_0)/sqrt(2), (x_0 - p_0)/sqrt(2)
    x_0_disp, p_0_disp = x_0 + ξ_d/λ/sqrt(2), p_0
    system = SystemMichiel(β, λ, x_0_disp, p_0_disp, 1)
    x_disp, p_disp, _ = simulator(t_max, system, drive, environment; t_save, equation = test_equation!)
    x = @.  x_disp- ξ_d/λ/sqrt(2)
    p = p_disp
    scatter!(x, p, label="", ms=2)
end
plot!(xlabel = L"x", ylabel= L"p",
title=L"\beta="*"$β")
display(p0)


# Lyapuov exponent estimation

In [ ]:
β = 0.5
λ = 0.6

ξ_d = 3.3
ν_d = 1.96

Q = Inf
t_max=1000.0

x_0 = .5
p_0 = .5

dt_save = 2pi/ν_d
t_save = 0.0 : dt_save :t_max

system = SystemMichiel(β, λ, x_0, p_0, 1)
environment = MeanField_Michiel(Q)
drive = DriveOneMode(ξ_d, ν_d, -pi/2)
x_1, p_1, _ = simulator(t_max, system, drive, environment; t_save)

system = SystemMichiel(β, λ, x_0+1e-6, p_0, 1)
x_2, p_2, t_list = simulator(t_max, system, drive, environment; t_save)
dist = sqrt.((x_1-x_2).^2+(p_1-p_2).^2)
plot(t_list, dist, yaxis=:log)

In [ ]:
β = 2.5
λ = 0.6

ξ_d = 3.3
ν_d = 1.96

Q = Inf
t_max=350.0

x_0 = .5
p_0 = .5

dt_save = 2pi/ν_d
t_save = 0.0 : dt_save :t_max

system = SystemMichiel(β, λ, x_0, p_0, 1)
environment = MeanField_Michiel(Q)
drive = DriveOneMode(ξ_d, ν_d, -pi/2)
x_1, p_1, _ = simulator(t_max, system, drive, environment; t_save)

system = SystemMichiel(β, λ, x_0+1e-6, p_0, 1)
x_2, p_2, t_list = simulator(t_max, system, drive, environment; t_save)
dist = sqrt.((x_1-x_2).^2+(p_1-p_2).^2)
plot(t_list, dist, yaxis=:log)